In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tarea3_DataFrames") \
    .master("local[*]") \
    .getOrCreate()

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/03 09:22:51 WARN Utils: Your hostname, DESKTOP-MA9NEBV, resolves to a loopback address: 127.0.1.1; using 172.23.214.117 instead (on interface eth0)
26/03/03 09:22:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 09:22:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import requests

url = "https://restcountries.com/v3.1/all?fields=name,population,region,area"
response = requests.get(url)
data = response.json()

In [3]:
datos = [
    (
        x["name"]["common"],
        x.get("region", "Unknown"),
        x.get("population", 0),
        x.get("area", 0)
    )
    for x in data
]

In [4]:
df = spark.createDataFrame(
    datos,
    ["pais", "region", "poblacion", "area"]
)

df.show(5)
df.printSchema()

[Stage 0:>                                                          (0 + 1) / 1]

+-----------+-------+---------+--------+
|       pais| region|poblacion|    area|
+-----------+-------+---------+--------+
|   Zimbabwe| Africa| 17073087|390757.0|
|   Kiribati|Oceania|   120740|   811.0|
|      Ghana| Africa| 33742380|238533.0|
|North Korea|   Asia| 25950000|120538.0|
|      Spain| Europe| 49315949|505992.0|
+-----------+-------+---------+--------+
only showing top 5 rows
root
 |-- pais: string (nullable = true)
 |-- region: string (nullable = true)
 |-- poblacion: long (nullable = true)
 |-- area: double (nullable = true)



In [5]:
df.select("pais", "poblacion").show(5)

+-----------+---------+
|       pais|poblacion|
+-----------+---------+
|   Zimbabwe| 17073087|
|   Kiribati|   120740|
|      Ghana| 33742380|
|North Korea| 25950000|
|      Spain| 49315949|
+-----------+---------+
only showing top 5 rows


In [6]:
df = df.withColumnRenamed("pais", "nombre_pais")
df.show(5)

+-----------+-------+---------+--------+
|nombre_pais| region|poblacion|    area|
+-----------+-------+---------+--------+
|   Zimbabwe| Africa| 17073087|390757.0|
|   Kiribati|Oceania|   120740|   811.0|
|      Ghana| Africa| 33742380|238533.0|
|North Korea|   Asia| 25950000|120538.0|
|      Spain| Europe| 49315949|505992.0|
+-----------+-------+---------+--------+
only showing top 5 rows


In [7]:
from pyspark.sql.functions import col

df = df.withColumn(
    "densidad",
    col("poblacion") / col("area")
)

df.select("nombre_pais", "densidad").show(5)

+-----------+------------------+
|nombre_pais|          densidad|
+-----------+------------------+
|   Zimbabwe| 43.69233820507374|
|   Kiribati|148.87792848335388|
|      Ghana|141.45791148394562|
|North Korea|215.28480645107766|
|      Spain| 97.46389073345033|
+-----------+------------------+
only showing top 5 rows


In [8]:
from pyspark.sql.functions import when

df = df.withColumn(
    "region",
    when(col("region").isNull(), "Sin region")
    .otherwise(col("region"))
)

In [9]:
df.filter(col("poblacion") > 50_000_000).show()

+--------------+--------+----------+-----------+------------------+
|   nombre_pais|  region| poblacion|       area|          densidad|
+--------------+--------+----------+-----------+------------------+
|        Brazil|Americas| 213421037|  8515767.0|25.061869001347734|
|      Colombia|Americas|  53057212|  1141748.0|46.470159790076266|
|         Italy|  Europe|  58927633|   301336.0|195.55457363209175|
|      Ethiopia|  Africa| 111652998|  1104300.0|101.10748709589785|
|       Vietnam|    Asia| 101343800|   331212.0| 305.9786481166141|
|        Russia|  Europe| 146028325|1.7098246E7| 8.540544158740024|
|       Myanmar|    Asia|  51316756|   676578.0|  75.8475090824709|
|         Egypt|  Africa| 107271260|  1002450.0|107.00908773504914|
|     Indonesia|    Asia| 284438782|  1904569.0|149.34548551404544|
| United States|Americas| 340110988|  9525067.0| 35.70693917428612|
|        Turkey|    Asia|  85664944|   783562.0|109.32758862731986|
|       Germany|  Europe|  83491249|   357114.0|

In [10]:
df.filter(
    (col("region") == "Asia") &
    (col("densidad") > 200)
).show()

+-----------+------+----------+---------+------------------+
|nombre_pais|region| poblacion|     area|          densidad|
+-----------+------+----------+---------+------------------+
|North Korea|  Asia|  25950000| 120538.0|215.28480645107766|
|    Vietnam|  Asia| 101343800| 331212.0| 305.9786481166141|
|     Israel|  Asia|  10134800|  21937.0| 461.9957150020513|
|  Palestine|  Asia|   5483450|   6220.0| 881.5836012861737|
|      Qatar|  Asia|   3173024|  11586.0|273.86708095977906|
|South Korea|  Asia|  51159889| 100210.0|510.52678375411637|
|   Maldives|  Asia|    515132|    300.0|1717.1066666666666|
|    Bahrain|  Asia|   1594654|    765.0|2084.5150326797384|
|Philippines|  Asia| 114123600| 342353.0| 333.3506643727381|
|      India|  Asia|1417492000|3287263.0| 431.2073600439028|
|     Taiwan|  Asia|  23317031|  36197.0| 644.1702627289554|
|  Sri Lanka|  Asia|  21763170|  65610.0| 331.7050754458162|
|  Hong Kong|  Asia|   7527500|   1104.0|  6818.38768115942|
|     Kuwait|  Asia|   4

In [11]:
df.orderBy(col("poblacion").desc()).show(10)

+-------------+--------+----------+-----------+------------------+
|  nombre_pais|  region| poblacion|       area|          densidad|
+-------------+--------+----------+-----------+------------------+
|        India|    Asia|1417492000|  3287263.0| 431.2073600439028|
|        China|    Asia|1408280000|  9706961.0|145.07939199508476|
|United States|Americas| 340110988|  9525067.0| 35.70693917428612|
|    Indonesia|    Asia| 284438782|  1904569.0|149.34548551404544|
|     Pakistan|    Asia| 241499431|   796095.0| 303.3550405416439|
|      Nigeria|  Africa| 223800000|   923768.0|242.26862155865976|
|       Brazil|Americas| 213421037|  8515767.0|25.061869001347734|
|   Bangladesh|    Asia| 169828911|   147570.0|1150.8362878633868|
|       Russia|  Europe| 146028325|1.7098246E7| 8.540544158740024|
|       Mexico|Americas| 130575786|  1964375.0| 66.47192414890232|
+-------------+--------+----------+-----------+------------------+
only showing top 10 rows


In [12]:
df.groupBy("region").avg("poblacion").show()

+---------+--------------------+
|   region|      avg(poblacion)|
+---------+--------------------+
|Antarctic|               340.0|
|   Europe|1.3993545698113207E7|
|   Africa|2.4787532389830507E7|
| Americas|      1.8617496125E7|
|  Oceania|  1779988.0740740742|
|     Asia|       9.449463932E7|
+---------+--------------------+

